# How tools eat the context window

**Business case:** MegaMart, an online electronics store. We are building an
operations assistant that support staff use to answer questions about
products, orders, inventory and customers.




In [3]:
# Cell 1 - setup

import json
import ollama

MODEL = "qwen3:8b"   # must be a tool-capable model

In [4]:
# Cell 2 - PRODUCT tools (5)
# Note: every docstring below becomes part of the schema,
# and every schema goes into the context on EVERY request.

def search_products(query: str, category: str = "all", max_results: int = 5):
    """Search the MegaMart product catalog by keyword.

    Use this when the user mentions a product by name or describes what they
    are looking for but you do not have a product ID yet. Always search first
    if you are not sure of the exact product ID, because every other product
    tool requires the ID.

    Args:
        query: Search keywords, for example 'wireless mouse' or 'AeroBook'.
        category: Filter by category. One of 'laptops', 'phones', 'audio',
            'accessories', 'monitors', or 'all'. Defaults to 'all'.
        max_results: Maximum number of products to return, between 1 and 20.

    Returns:
        A list of matching products. Each product has: product_id (str),
        name (str), category (str), price_inr (int), rating (float),
        in_catalog_since (str, ISO date).
    """
    return [
        {"product_id": "P-1001", "name": "AeroBook 14", "category": "laptops",
         "price_inr": 94990, "rating": 4.4, "in_catalog_since": "2024-08-01"},
        {"product_id": "P-1002", "name": "AeroBook 16 Pro", "category": "laptops",
         "price_inr": 164990, "rating": 4.6, "in_catalog_since": "2024-11-15"},
    ][:max_results]


def get_product_details(product_id: str):
    """Get the full specification sheet for one product.

    Use this after you have a product_id from search_products. Do not guess
    specifications from memory - always retrieve them, because product
    revisions change hardware without changing the product name.

    Args:
        product_id: The MegaMart product ID, for example 'P-1001'.

    Returns:
        A dict with: product_id, name, brand, category, price_inr, cpu, ram,
        storage, display, battery, weight_kg, ports (list of str),
        warranty_months (int), box_contents (list of str).
    """
    return {
        "product_id": product_id, "name": "AeroBook 14", "brand": "Aero",
        "category": "laptops", "price_inr": 94990,
        "cpu": "8-core 3.2GHz", "ram": "16GB DDR5", "storage": "512GB NVMe",
        "display": "14in 2880x1800 120Hz", "battery": "72Wh, 18h video",
        "weight_kg": 1.24, "ports": ["2x USB-C", "1x USB-A", "HDMI 2.1"],
        "warranty_months": 12, "box_contents": ["laptop", "65W charger", "manual"],
    }


def get_price_history(product_id: str, months: int = 6):
    """Get the month-by-month price history of a product.

    Use this when a customer claims the price was lower earlier, when staff
    need to check discount eligibility, or when deciding whether a current
    price is a genuine drop. Prices are recorded on the 1st of each month.

    Args:
        product_id: The MegaMart product ID.
        months: How many months of history to return, between 1 and 24.

    Returns:
        A list of entries, newest first. Each entry has: month (str, YYYY-MM),
        price_inr (int), was_on_promotion (bool).
    """
    return [
        {"month": "2026-07", "price_inr": 94990, "was_on_promotion": False},
        {"month": "2026-06", "price_inr": 99990, "was_on_promotion": False},
        {"month": "2026-05", "price_inr": 89990, "was_on_promotion": True},
    ][:months]


def check_warranty_info(product_id: str, purchase_date: str):
    """Check whether a product is still under warranty.

    Use this when a customer reports a defect or asks about repair coverage.
    Warranty is calculated from the purchase date, not the delivery date.

    Args:
        product_id: The MegaMart product ID.
        purchase_date: Date of purchase in YYYY-MM-DD format. Take this from
            the order record, not from the customer's memory.

    Returns:
        A dict with: product_id, warranty_months (int), covered_until
        (str, YYYY-MM-DD), still_covered (bool), covers (list of str),
        does_not_cover (list of str).
    """
    return {
        "product_id": product_id, "warranty_months": 12,
        "covered_until": "2026-11-03", "still_covered": True,
        "covers": ["manufacturing defects", "battery below 80% health"],
        "does_not_cover": ["physical damage", "liquid damage"],
    }


def get_product_reviews(product_id: str):
    """Get all customer reviews for a product, with full review text.

    Use this when someone asks what customers think of a product, or when
    investigating a complaint pattern. Returns every review in full.

    Args:
        product_id: The MegaMart product ID.

    Returns:
        A list of reviews. Each review has: reviewer (str), rating (int, 1-5),
        date (str), title (str), full_text (str), verified_purchase (bool),
        helpful_votes (int).
    """
    # THE VERBOSE ONE. A real reviews API returns everything, so ours does too.
    base_reviews = [
        (5, "Excellent for the price",
         "I have been using this laptop for three months now and it has exceeded "
         "my expectations in almost every way. The display is bright and sharp, "
         "the keyboard is comfortable for long typing sessions, and the battery "
         "genuinely lasts a full working day. I regularly run a code editor, "
         "about twenty browser tabs, and a music app at the same time and have "
         "not noticed any slowdown. The speakers are average but acceptable. "
         "Shipping was fast and the packaging was secure."),
        (4, "Very good, minor heat issue",
         "Overall a very solid machine. Performance is great for development "
         "work and the screen is one of the best I have used at this price. My "
         "only complaint is that the underside gets noticeably warm when "
         "compiling large projects or on video calls longer than an hour. It is "
         "never uncomfortable on a desk but you would notice it on your lap. "
         "Battery life is as advertised. Would still recommend it to a friend."),
        (2, "Disappointed with the speakers and support",
         "The laptop itself is fine for basic use but I had a frustrating "
         "experience. The speakers crackle at high volume, which I first "
         "noticed on video calls. I contacted support and it took three "
         "separate chats over two weeks to get a pickup arranged. The "
         "replacement unit works properly, so I am keeping it, but the process "
         "soured the experience for me considerably. Check your speakers "
         "immediately after unboxing."),
        (5, "Perfect for students",
         "Bought this for my daughter who is studying design. It handles her "
         "editing software without any trouble and it is light enough to carry "
         "to college every day. The build quality feels far more premium than "
         "the price suggests. She particularly likes the fingerprint reader and "
         "the fact that it charges from a normal USB-C power bank in an "
         "emergency. Great value overall and we would buy from this brand again."),
        (3, "Average battery on my unit",
         "Mixed feelings. The screen and keyboard are genuinely excellent, but "
         "my unit gets closer to nine hours of battery than the eighteen "
         "advertised, even with conservative settings. Maybe I received a weak "
         "battery, maybe the claim is optimistic. Performance is otherwise fine "
         "for office work and light photo editing. I would rate it higher if "
         "the battery matched the marketing."),
    ]
    reviewers = ["Priya S", "Rahul M", "Anita K", "Vikram J", "Sneha P",
                 "Arjun T", "Divya R", "Karan B", "Meera L", "Sanjay G"]
    reviews = []
    for i in range(25):
        rating, title, text = base_reviews[i % len(base_reviews)]
        reviews.append({
            "reviewer": reviewers[i % len(reviewers)],
            "rating": rating,
            "date": f"2026-{(i % 6) + 1:02d}-{(i % 27) + 1:02d}",
            "title": title,
            "full_text": text,
            "verified_purchase": i % 4 != 0,
            "helpful_votes": (i * 7) % 40,
        })
    return reviews

In [5]:
# Cell 3 - INVENTORY tools (4)

def get_inventory_level(product_id: str, warehouse: str = "all"):
    """Get current stock levels for a product.

    Use this for any question about availability or stock. Stock numbers
    update every 15 minutes, so treat them as near-real-time.

    Args:
        product_id: The MegaMart product ID.
        warehouse: One of 'delhi', 'mumbai', 'bangalore', or 'all'.

    Returns:
        A dict with: product_id, total_units (int), by_warehouse (dict of
        warehouse name to units), units_reserved_for_orders (int),
        restock_expected (str date or null).
    """
    return {
        "product_id": product_id, "total_units": 47,
        "by_warehouse": {"delhi": 12, "mumbai": 30, "bangalore": 5},
        "units_reserved_for_orders": 8, "restock_expected": None,
    }


def get_low_stock_items(category: str = "all", threshold: int = 10):
    """List products whose total stock has fallen below a threshold.

    Use this for restocking decisions and for warning staff before promising
    delivery dates on scarce items.

    Args:
        category: Filter by category, or 'all'.
        threshold: Stock level below which an item counts as low. Default 10.

    Returns:
        A list of items, each with: product_id, name, total_units (int),
        daily_sales_rate (float), days_until_stockout (int).
    """
    return [
        {"product_id": "P-1003", "name": "Vertex 14", "total_units": 2,
         "daily_sales_rate": 1.5, "days_until_stockout": 1},
        {"product_id": "P-2044", "name": "SoundCore Buds", "total_units": 7,
         "daily_sales_rate": 3.2, "days_until_stockout": 2},
    ]


def get_supplier_info(product_id: str):
    """Get the supplier and lead time for a product.

    Use this when planning restocks or when a customer asks how long an
    out-of-stock item will take to return.

    Args:
        product_id: The MegaMart product ID.

    Returns:
        A dict with: supplier_name (str), lead_time_days (int),
        minimum_order_units (int), last_order_date (str).
    """
    return {
        "supplier_name": "Aero Distribution India", "lead_time_days": 9,
        "minimum_order_units": 20, "last_order_date": "2026-06-28",
    }


def create_restock_order(product_id: str, units: int):
    """Create a restock order with the product's supplier.

    Use this only after checking get_inventory_level and get_supplier_info.
    Orders below the supplier's minimum_order_units will be rejected.

    Args:
        product_id: The MegaMart product ID.
        units: Number of units to order. Must meet the supplier minimum.

    Returns:
        A dict with: restock_order_id (str), status (str), expected_arrival
        (str, YYYY-MM-DD).
    """
    return {"restock_order_id": "RS-7781", "status": "placed",
            "expected_arrival": "2026-07-31"}

In [6]:
# Cell 4 - ORDER tools (5)

def get_order_details(order_id: str):
    """Get the full details of one order.

    Use this whenever a customer question mentions an order number. The order
    record is the source of truth for what was bought, when, and for how much.

    Args:
        order_id: The order ID, format 'ORD-' followed by digits.

    Returns:
        A dict with: order_id, customer_id, order_date (str), items (list of
        dicts with product_id, name, quantity, price_inr), total_inr (int),
        payment_method (str), status (str: one of 'placed', 'shipped',
        'delivered', 'cancelled', 'returned').
    """
    return {
        "order_id": order_id, "customer_id": "C-5501",
        "order_date": "2026-07-10",
        "items": [{"product_id": "P-1001", "name": "AeroBook 14",
                    "quantity": 1, "price_inr": 94990}],
        "total_inr": 94990, "payment_method": "UPI", "status": "delivered",
    }


def list_recent_orders(customer_id: str, limit: int = 5):
    """List a customer's most recent orders, newest first.

    Use this when the customer does not know their order number, or when you
    need to see their purchase pattern.

    Args:
        customer_id: The customer ID, format 'C-' followed by digits.
        limit: How many orders to return, between 1 and 20.

    Returns:
        A list of orders, each with: order_id, order_date, total_inr, status.
    """
    return [
        {"order_id": "ORD-90112", "order_date": "2026-07-10",
         "total_inr": 94990, "status": "delivered"},
        {"order_id": "ORD-88437", "order_date": "2026-05-22",
         "total_inr": 2499, "status": "delivered"},
    ][:limit]


def get_shipping_status(order_id: str):
    """Get the live shipping status and tracking events for an order.

    Use this for any 'where is my order' question. Events are newest first.

    Args:
        order_id: The order ID.

    Returns:
        A dict with: order_id, courier (str), tracking_id (str),
        current_status (str), expected_delivery (str, YYYY-MM-DD),
        events (list of dicts with timestamp and description).
    """
    return {
        "order_id": order_id, "courier": "BlueDart", "tracking_id": "BD-4471822",
        "current_status": "out for delivery",
        "expected_delivery": "2026-07-22",
        "events": [
            {"timestamp": "2026-07-22 08:10", "description": "Out for delivery, Gurugram hub"},
            {"timestamp": "2026-07-21 19:44", "description": "Arrived at Gurugram hub"},
        ],
    }


def process_refund(order_id: str, reason: str):
    """Initiate a refund for a delivered order.

    Use this only after confirming the order status is 'delivered' or
    'returned' and the request is within the 10-day return window. Refunds go
    back to the original payment method within 5 to 7 working days.

    Args:
        order_id: The order ID to refund.
        reason: Short reason code. One of 'defective', 'wrong_item',
            'not_as_described', 'changed_mind'.

    Returns:
        A dict with: refund_id (str), order_id, amount_inr (int),
        status (str), expected_completion (str, YYYY-MM-DD).
    """
    return {"refund_id": "RF-3320", "order_id": order_id,
            "amount_inr": 94990, "status": "initiated",
            "expected_completion": "2026-07-29"}


def get_return_requests(status: str = "pending"):
    """List product return requests by status.

    Use this for questions about pending returns, or to check whether a
    specific customer already has a return in progress.

    Args:
        status: One of 'pending', 'approved', 'rejected', 'completed', 'all'.

    Returns:
        A list of requests, each with: return_id, order_id, customer_id,
        product_id, reason (str), requested_on (str), status (str).
    """
    return [
        {"return_id": "RT-1188", "order_id": "ORD-90112",
         "customer_id": "C-5501", "product_id": "P-1001",
         "reason": "defective", "requested_on": "2026-07-18",
         "status": "pending"},
    ]

In [7]:
# Cell 5 - CUSTOMER and ANALYTICS tools (4)

def get_customer_profile(customer_id: str):
    """Get a customer's profile and account standing.

    Use this to check membership tier, contact details, and whether the
    account has any flags before making exceptions for them.

    Args:
        customer_id: The customer ID.

    Returns:
        A dict with: customer_id, name (str), email (str), city (str),
        member_since (str), tier (str: 'standard', 'gold', 'platinum'),
        lifetime_spend_inr (int), account_flags (list of str).
    """
    return {
        "customer_id": customer_id, "name": "Rohit Verma",
        "email": "rohit.v@example.com", "city": "Gurugram",
        "member_since": "2023-02-14", "tier": "gold",
        "lifetime_spend_inr": 287400, "account_flags": [],
    }


def create_support_ticket(customer_id: str, subject: str, priority: str = "normal"):
    """Create a support ticket for an issue that cannot be resolved in chat.

    Use this when the issue needs a human specialist: hardware inspection,
    billing disputes, or anything requiring physical logistics beyond a
    standard return.

    Args:
        customer_id: The customer ID.
        subject: One-line summary of the issue, under 100 characters.
        priority: One of 'low', 'normal', 'high'. Use 'high' only for
            payment failures or safety issues.

    Returns:
        A dict with: ticket_id (str), status (str), expected_first_response
        (str, e.g. 'within 4 hours').
    """
    return {"ticket_id": "TK-20991", "status": "open",
            "expected_first_response": "within 4 hours"}


def get_sales_summary(period: str = "last_7_days", category: str = "all"):
    """Get aggregated sales figures for a period.

    Use this for business questions about revenue, units sold, and top
    sellers. Figures exclude cancelled orders and include GST.

    Args:
        period: One of 'today', 'last_7_days', 'last_30_days', 'this_quarter'.
        category: Filter by category, or 'all'.

    Returns:
        A dict with: period, total_revenue_inr (int), units_sold (int),
        average_order_value_inr (int), top_products (list of dicts with
        product_id, name, units).
    """
    return {
        "period": period, "total_revenue_inr": 4823000, "units_sold": 214,
        "average_order_value_inr": 22537,
        "top_products": [
            {"product_id": "P-1001", "name": "AeroBook 14", "units": 31},
            {"product_id": "P-2044", "name": "SoundCore Buds", "units": 58},
        ],
    }


def get_payment_status(order_id: str):
    """Check the payment status of an order.

    Use this when a customer says they were charged but the order shows
    unpaid, or when a refund has not arrived. Payment records update within
    two minutes of the bank confirmation.

    Args:
        order_id: The order ID.

    Returns:
        A dict with: order_id, amount_inr (int), method (str), status (str:
        'paid', 'pending', 'failed', 'refunded'), transaction_id (str),
        paid_at (str timestamp or null).
    """
    return {"order_id": order_id, "amount_inr": 94990, "method": "UPI",
            "status": "paid", "transaction_id": "TXN-88123471",
            "paid_at": "2026-07-10 14:22"}


# All 18 tools in one list. This is what we hand to Ollama.
# No schemas anywhere - Ollama builds them from the docstrings and type hints.
TOOL_FUNCTIONS = [
    search_products, get_product_details, get_price_history,
    check_warranty_info, get_product_reviews,
    get_inventory_level, get_low_stock_items, get_supplier_info,
    create_restock_order,
    get_order_details, list_recent_orders, get_shipping_status,
    process_refund, get_return_requests,
    get_customer_profile, create_support_ticket, get_sales_summary,
    get_payment_status,
]

REGISTRY = {f.__name__: f for f in TOOL_FUNCTIONS}
print(len(TOOL_FUNCTIONS), "tools ready")

18 tools ready


In [8]:
print(REGISTRY)

{'search_products': <function search_products at 0x000001DEF5B6EC00>, 'get_product_details': <function get_product_details at 0x000001DEF8C48400>, 'get_price_history': <function get_price_history at 0x000001DEF8C48220>, 'check_warranty_info': <function check_warranty_info at 0x000001DEF8C484A0>, 'get_product_reviews': <function get_product_reviews at 0x000001DEF8C482C0>, 'get_inventory_level': <function get_inventory_level at 0x000001DEF5B6F2E0>, 'get_low_stock_items': <function get_low_stock_items at 0x000001DEF8C48860>, 'get_supplier_info': <function get_supplier_info at 0x000001DEF8C48180>, 'create_restock_order': <function create_restock_order at 0x000001DEF8C48B80>, 'get_order_details': <function get_order_details at 0x000001DEF8C480E0>, 'list_recent_orders': <function list_recent_orders at 0x000001DEF8C49080>, 'get_shipping_status': <function get_shipping_status at 0x000001DEF8C48680>, 'process_refund': <function process_refund at 0x000001DEF8C48F40>, 'get_return_requests': <func

In [10]:
# Cell 7 - the system prompt
# Behaviour only. No tool names, no call formats - the tools parameter
# handles all of that.

SYSTEM_PROMPT = """You are the operations assistant for MegaMart, an online \
electronics store. Support staff use you to answer questions about products, \
orders, inventory and customers.

Never state product data, stock, prices or order details from memory. Always \
retrieve current data before answering.

Lead with the direct answer, then supporting detail. Keep answers under 100 \
words unless a comparison or report was requested.

If a request is out of scope (billing disputes, legal issues), create a \
support ticket rather than guessing."""

In [11]:
 messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "HI,how are you"},
    ]

response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=TOOL_FUNCTIONS
        )

In [12]:
print(response)

model='qwen3:8b' created_at='2026-07-22T11:37:46.6610609Z' done=True done_reason='stop' total_duration=18372717500 load_duration=7172651500 prompt_eval_count=2109 prompt_eval_duration=1935189000 eval_count=165 eval_duration=9224817000 message=Message(role='assistant', content="Hello! I'm just a chatbot, but I'm here and ready to help with your MegaMart queries. How can I assist you today? 😊", thinking='Okay, the user said, "HI, how are you." I need to respond appropriately. Let me check the tools provided. Wait, the tools are for functions related to product searches, order details, inventory, etc. But this is a general greeting. The user isn\'t asking for any specific information or action. So, I don\'t need to call any functions here. My role is to greet them and offer assistance. Let me make sure there\'s no hidden request in their message. Nope, it\'s just a greeting. So, I\'ll respond with a friendly message and ask how I can help. No tool calls required.\n', images=None, tool_nam

In [13]:
print(response.message.content)

Hello! I'm just a chatbot, but I'm here and ready to help with your MegaMart queries. How can I assist you today? 😊


In [14]:
print(response.prompt_eval_count)

2109


In [15]:
# 1. Initialize the messages list outside the function with just the system prompt
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def run(question, max_turns=10):
    # 2. Append the new user question to the existing external list
    messages.append({"role": "user", "content": question})

    for turn in range(1, max_turns + 1):
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=TOOL_FUNCTIONS,
            options={"num_ctx": 8000}
        )

        # the size of the ENTIRE context the model just read
        size = response.prompt_eval_count or 0
        print(f"turn {turn:>2}: {size:>6} tokens  {'#' * (size // 200)}")

        messages.append(response.message)

        if not response.message.tool_calls:
            print(f"\nANSWER: {response.message.content}\n")
            break

        for call in response.message.tool_calls:
            name = call.function.name
            args = call.function.arguments
            print(f"           -> {name}({args})")

            if name in REGISTRY:
                result = REGISTRY[name](**args)
            else:
                result = {"error": f"no tool named '{name}'"}

            # the line that grows the context - results are appended, never removed
            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(result),
            })
    else:
        print("stopped: hit max_turns")

    print(f"messages in history: {len(messages)}")
    return messages

In [ ]:
###### Cell 10 - RUN 2: a question that triggers the verbose tool
# get_product_reviews returns 25 full-text reviews. Watch what one single
# tool result does to the context between turn 1 and turn 2.

h2 = run("What are customers saying about the AeroBook 14? "
         "Summarise the common complaints.")

In [17]:
messages

[{'role': 'system',
  'content': 'You are the operations assistant for MegaMart, an online electronics store. Support staff use you to answer questions about products, orders, inventory and customers.\n\nNever state product data, stock, prices or order details from memory. Always retrieve current data before answering.\n\nLead with the direct answer, then supporting detail. Keep answers under 100 words unless a comparison or report was requested.\n\nIf a request is out of scope (billing disputes, legal issues), create a support ticket rather than guessing.'},
 {'role': 'user',
  'content': 'What are customers saying about the AeroBook 14? Summarise the common complaints.'},
 Message(role='assistant', content='', thinking='Okay, the user is asking about customer reviews for the AeroBook 14 and wants a summary of common complaints. First, I need to find the product ID for the AeroBook 14. Since the user mentioned the product name, I should use the search_products function with the query 